# 13 - Hard-negative retraining of the hazard detector (Kaggle)

**What it does.** The shipped detector (v2) sometimes calls safe ground a hazard with confidence. In flight those false alarms become needless back-aways (notebook 11: 25 of 300 flights, against 4 with no detector). A *hard negative* is a frame where that happens. This notebook finds the hard negatives (frames of the training set and of 2,000 fresh low-altitude synthetic frames where v2 is confidently wrong on safe ground), then fine-tunes v2 on them three ways from the same start: `ctrl` (the same number of frames per split, chosen at random: the control), `hn_frames` (hard frames oversampled) and `hn_pixel` (oversampling plus a loss on the hardest safe pixels of each batch). Every variant is scored on the same held-out east-half real frames and compared at matched hazard recall.

**Extension, not the mission.** The real OSIRIS-REx had no onboard detector.

Settings: **GPU T4 x2**, **Internet on**, add the private dataset `touchdown-detector-v2` as an input. Save Version, Save & Run All (Commit). Roughly 3 hours. Output: `hn_run/` (checkpoints, evals, `compare.md`, `chosen.txt`). Notebook 14 takes this notebook's output as input and tests the chosen variant in the closed loop.

In [ ]:
!git clone -q https://github.com/aryanyaksh-art/touch_down.git /tmp/touch_down || (cd /tmp/touch_down && git pull -q)
!sh /tmp/touch_down/scripts/kaggle_setup.sh 2>&1 | tail -5

In [ ]:
import os
# the v2 detector was uploaded as two parts; join them back
!find /kaggle/input -name "best.pt.part*" | sort | xargs cat > /tmp/best.pt
assert os.path.getsize('/tmp/best.pt') == 17520863, os.path.getsize('/tmp/best.pt')

In [ ]:
# prints a line per step; the training logs are in /kaggle/working/hn_run/train_*.log
!python /tmp/touch_down/scripts/kaggle_hardneg.py
!cat /kaggle/working/hn_run/compare.md /kaggle/working/hn_run/mine.log | tail -40
!tail -3 /kaggle/working/hn_run/train_ctrl.log /kaggle/working/hn_run/train_hn_frames.log /kaggle/working/hn_run/train_hn_pixel.log